# 🔥 Drone Swarm Fire Control - RL Training Notebook

**Google Colab Pro Training Environment**

This notebook trains reinforcement learning agents to coordinate drone swarms for wildfire suppression.

## Features
- PPO, DQN, SAC, A2C algorithm support
- GPU-accelerated training
- Real-time training visualization
- Model checkpointing and evaluation
- Curriculum learning support

---

## 1. Setup Environment

In [ ]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab")
    
    # Check GPU availability
    import torch
    if torch.cuda.is_available():
        print(f"GPU Available: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")
else:
    print("Running locally")

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3[extra] tensorboard scipy matplotlib numpy pyyaml tqdm

In [ ]:
# Clone repository (if on Colab)
if IN_COLAB:
    !git clone https://github.com/ScottMccffry/fire_control.git 2>/dev/null || (cd fire_control && git pull)
    %cd fire_control
else:
    # Local development - adjust path as needed
    import os
    os.chdir('/home/user/fire_control')

In [ ]:
# Import libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import clear_output, display
import warnings
warnings.filterwarnings('ignore')

# Add project to path
sys.path.insert(0, os.getcwd())

# Import fire control modules
from envs.fire_control_env import FireControlEnv
from config import Config, load_config
from sim.mock_fire import MockFireSimulator
from drones.swarm import DroneSwarm

# Import RL libraries
import torch
from stable_baselines3 import PPO, DQN, A2C, SAC
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, BaseCallback
from stable_baselines3.common.utils import set_random_seed

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
# Training Configuration
CONFIG = {
    # Environment settings
    'swarm_size': 30,           # Number of drones (reduce for faster training)
    'grid_size': (40, 40),      # Fire grid size (reduce for faster training)
    'max_episode_steps': 150,   # Steps per episode
    
    # Algorithm selection
    'algorithm': 'PPO',         # PPO, DQN, A2C, or SAC
    
    # Training settings
    'total_timesteps': 500_000, # Total training steps (increase for better results)
    'n_envs': 4,                # Parallel environments
    'eval_freq': 10_000,        # Evaluation frequency
    'save_freq': 25_000,        # Checkpoint frequency
    
    # PPO Hyperparameters
    'learning_rate': 3e-4,
    'n_steps': 2048,
    'batch_size': 64,
    'n_epochs': 10,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'ent_coef': 0.01,
    
    # Paths
    'log_dir': 'logs/',
    'checkpoint_dir': 'checkpoints/',
    'seed': 42,
}

# Create directories
os.makedirs(CONFIG['log_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

# Set device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# Set random seed
set_random_seed(CONFIG['seed'])

## 3. Environment Setup

In [ ]:
def make_env(rank, seed=0):
    """
    Create a wrapped environment instance.
    """
    def _init():
        env = FireControlEnv(
            swarm_size=CONFIG['swarm_size'],
            grid_size=CONFIG['grid_size'],
            max_episode_steps=CONFIG['max_episode_steps'],
        )
        env = Monitor(env)
        env.reset(seed=seed + rank)
        return env
    return _init

# Create vectorized training environment
print(f"Creating {CONFIG['n_envs']} parallel environments...")
env_fns = [make_env(i, CONFIG['seed']) for i in range(CONFIG['n_envs'])]
train_env = DummyVecEnv(env_fns)

# Wrap with VecNormalize for better training stability
train_env = VecNormalize(
    train_env,
    norm_obs=True,
    norm_reward=True,
    clip_obs=10.0,
    clip_reward=10.0,
)

# Create evaluation environment (not normalized for fair comparison)
eval_env = DummyVecEnv([make_env(0, CONFIG['seed'] + 1000)])

print(f"Observation space: {train_env.observation_space.shape}")
print(f"Action space: {train_env.action_space}")
print("Environments created successfully!")

## 4. Test Environment

In [ ]:
# Quick environment test
print("Testing environment with random actions...")

test_env = FireControlEnv(
    swarm_size=CONFIG['swarm_size'],
    grid_size=CONFIG['grid_size'],
    max_episode_steps=50,
)

obs, info = test_env.reset(seed=42)
print(f"Initial observation shape: {obs.shape}")
print(f"Initial info: {info}")

total_reward = 0
for step in range(20):
    action = test_env.action_space.sample()
    obs, reward, terminated, truncated, info = test_env.step(action)
    total_reward += reward
    
    if step % 5 == 0:
        print(f"Step {step}: Reward={reward:.2f}, Active fires={info['active_fires']}, Drones={info['operational_drones']}")
    
    if terminated or truncated:
        break

print(f"\nTotal reward after {step+1} steps: {total_reward:.2f}")
test_env.close()

## 5. Custom Training Callback

In [ ]:
class TrainingVisualizationCallback(BaseCallback):
    """
    Custom callback for real-time training visualization in Colab.
    """
    
    def __init__(self, plot_freq=5000, verbose=0):
        super().__init__(verbose)
        self.plot_freq = plot_freq
        self.episode_rewards = []
        self.episode_lengths = []
        self.episode_fires = []
        self.timesteps = []
        
    def _on_step(self):
        # Collect episode info
        for info in self.locals.get('infos', []):
            if info and 'episode' in info:
                self.episode_rewards.append(info['episode']['r'])
                self.episode_lengths.append(info['episode']['l'])
                self.timesteps.append(self.num_timesteps)
                
                if 'active_fires' in info:
                    self.episode_fires.append(info.get('active_fires', 0))
        
        # Plot periodically
        if self.num_timesteps % self.plot_freq == 0 and len(self.episode_rewards) > 0:
            self._plot_progress()
            
        return True
    
    def _plot_progress(self):
        clear_output(wait=True)
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # Episode rewards
        ax1 = axes[0]
        ax1.plot(self.episode_rewards, alpha=0.3, color='blue')
        if len(self.episode_rewards) >= 10:
            window = min(50, len(self.episode_rewards))
            smoothed = np.convolve(self.episode_rewards, np.ones(window)/window, mode='valid')
            ax1.plot(range(window-1, len(self.episode_rewards)), smoothed, color='blue', linewidth=2)
        ax1.set_xlabel('Episode')
        ax1.set_ylabel('Reward')
        ax1.set_title(f'Episode Rewards (n={len(self.episode_rewards)})')
        ax1.grid(True, alpha=0.3)
        
        # Episode lengths
        ax2 = axes[1]
        ax2.plot(self.episode_lengths, alpha=0.3, color='green')
        if len(self.episode_lengths) >= 10:
            window = min(50, len(self.episode_lengths))
            smoothed = np.convolve(self.episode_lengths, np.ones(window)/window, mode='valid')
            ax2.plot(range(window-1, len(self.episode_lengths)), smoothed, color='green', linewidth=2)
        ax2.set_xlabel('Episode')
        ax2.set_ylabel('Length')
        ax2.set_title('Episode Lengths')
        ax2.grid(True, alpha=0.3)
        
        # Training progress
        ax3 = axes[2]
        progress = self.num_timesteps / CONFIG['total_timesteps'] * 100
        ax3.barh(['Progress'], [progress], color='purple')
        ax3.barh(['Progress'], [100-progress], left=[progress], color='lightgray')
        ax3.set_xlim(0, 100)
        ax3.set_xlabel('Percent')
        ax3.set_title(f'Training Progress: {self.num_timesteps:,} / {CONFIG["total_timesteps"]:,}')
        
        # Stats text
        if len(self.episode_rewards) >= 10:
            recent_mean = np.mean(self.episode_rewards[-10:])
            recent_std = np.std(self.episode_rewards[-10:])
            ax3.text(50, 0.5, f'Recent Reward: {recent_mean:.1f} ± {recent_std:.1f}', 
                    ha='center', va='center', fontsize=10,
                    transform=ax3.transAxes)
        
        plt.tight_layout()
        plt.show()
        
        # Print stats
        print(f"\nTimestep: {self.num_timesteps:,} | Episodes: {len(self.episode_rewards)}")
        if len(self.episode_rewards) > 0:
            print(f"Mean Reward (last 10): {np.mean(self.episode_rewards[-10:]):.2f}")
            print(f"Best Reward: {max(self.episode_rewards):.2f}")

## 6. Create RL Model

In [ ]:
# Select algorithm
ALGORITHMS = {
    'PPO': PPO,
    'DQN': DQN,
    'A2C': A2C,
    'SAC': SAC,
}

algorithm_class = ALGORITHMS[CONFIG['algorithm']]
print(f"Using algorithm: {CONFIG['algorithm']}")

# Create model with appropriate hyperparameters
if CONFIG['algorithm'] == 'PPO':
    model = PPO(
        'MlpPolicy',
        train_env,
        learning_rate=CONFIG['learning_rate'],
        n_steps=CONFIG['n_steps'],
        batch_size=CONFIG['batch_size'],
        n_epochs=CONFIG['n_epochs'],
        gamma=CONFIG['gamma'],
        gae_lambda=CONFIG['gae_lambda'],
        clip_range=CONFIG['clip_range'],
        ent_coef=CONFIG['ent_coef'],
        verbose=0,
        device=DEVICE,
        tensorboard_log=CONFIG['log_dir'],
        seed=CONFIG['seed'],
    )
elif CONFIG['algorithm'] == 'DQN':
    model = DQN(
        'MlpPolicy',
        train_env,
        learning_rate=1e-4,
        buffer_size=100000,
        learning_starts=1000,
        batch_size=32,
        gamma=CONFIG['gamma'],
        verbose=0,
        device=DEVICE,
        tensorboard_log=CONFIG['log_dir'],
        seed=CONFIG['seed'],
    )
elif CONFIG['algorithm'] == 'A2C':
    model = A2C(
        'MlpPolicy',
        train_env,
        learning_rate=7e-4,
        n_steps=5,
        gamma=CONFIG['gamma'],
        ent_coef=CONFIG['ent_coef'],
        verbose=0,
        device=DEVICE,
        tensorboard_log=CONFIG['log_dir'],
        seed=CONFIG['seed'],
    )
else:  # SAC
    # SAC requires continuous action space, so we need to modify environment
    print("Note: SAC requires continuous actions. Using PPO instead.")
    model = PPO(
        'MlpPolicy',
        train_env,
        learning_rate=CONFIG['learning_rate'],
        verbose=0,
        device=DEVICE,
        tensorboard_log=CONFIG['log_dir'],
        seed=CONFIG['seed'],
    )

print(f"Model created on device: {model.device}")
print(f"Policy architecture: {model.policy}")

## 7. Setup Callbacks

In [ ]:
# Create callbacks
callbacks = []

# Checkpoint callback - save model periodically
checkpoint_callback = CheckpointCallback(
    save_freq=CONFIG['save_freq'] // CONFIG['n_envs'],
    save_path=CONFIG['checkpoint_dir'],
    name_prefix='fire_control',
    save_replay_buffer=False,
    save_vecnormalize=True,
)
callbacks.append(checkpoint_callback)

# Evaluation callback - evaluate and save best model
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(CONFIG['checkpoint_dir'], 'best'),
    log_path=os.path.join(CONFIG['log_dir'], 'eval'),
    eval_freq=CONFIG['eval_freq'] // CONFIG['n_envs'],
    n_eval_episodes=5,
    deterministic=True,
)
callbacks.append(eval_callback)

# Visualization callback
viz_callback = TrainingVisualizationCallback(plot_freq=5000)
callbacks.append(viz_callback)

print(f"Callbacks configured: {len(callbacks)} callbacks")

## 8. Train Model

In [ ]:
# Start training
print("="*60)
print(f"Starting Training - {CONFIG['algorithm']}")
print(f"Total timesteps: {CONFIG['total_timesteps']:,}")
print(f"Environment: {CONFIG['swarm_size']} drones, {CONFIG['grid_size']} grid")
print("="*60)

start_time = datetime.now()

try:
    model.learn(
        total_timesteps=CONFIG['total_timesteps'],
        callback=callbacks,
        progress_bar=False,  # We use our own visualization
    )
except KeyboardInterrupt:
    print("\nTraining interrupted by user")

end_time = datetime.now()
training_duration = end_time - start_time

print("\n" + "="*60)
print(f"Training Complete!")
print(f"Duration: {training_duration}")
print(f"Episodes completed: {len(viz_callback.episode_rewards)}")
if len(viz_callback.episode_rewards) > 0:
    print(f"Final mean reward: {np.mean(viz_callback.episode_rewards[-10:]):.2f}")
print("="*60)

## 9. Save Final Model

In [ ]:
# Save final model
final_model_path = os.path.join(CONFIG['checkpoint_dir'], 'final_model')
model.save(final_model_path)
print(f"Model saved to: {final_model_path}.zip")

# Save VecNormalize statistics
vec_norm_path = os.path.join(CONFIG['checkpoint_dir'], 'vec_normalize.pkl')
train_env.save(vec_norm_path)
print(f"VecNormalize saved to: {vec_norm_path}")

# Save training config
import json
config_path = os.path.join(CONFIG['checkpoint_dir'], 'training_config.json')
with open(config_path, 'w') as f:
    save_config = CONFIG.copy()
    save_config['grid_size'] = list(save_config['grid_size'])
    save_config['training_duration'] = str(training_duration)
    save_config['final_episodes'] = len(viz_callback.episode_rewards)
    json.dump(save_config, f, indent=2)
print(f"Config saved to: {config_path}")

## 10. Evaluate Trained Model

In [ ]:
def evaluate_model(model, env, n_episodes=10, deterministic=True):
    """
    Evaluate model performance.
    """
    episode_rewards = []
    episode_lengths = []
    episode_infos = []
    
    for ep in range(n_episodes):
        obs = env.reset()
        done = False
        episode_reward = 0
        episode_length = 0
        
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, done, info = env.step(action)
            episode_reward += reward[0]
            episode_length += 1
        
        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        episode_infos.append(info[0] if info else {})
        
        print(f"Episode {ep+1}: Reward={episode_reward:.2f}, Length={episode_length}")
    
    return {
        'mean_reward': np.mean(episode_rewards),
        'std_reward': np.std(episode_rewards),
        'mean_length': np.mean(episode_lengths),
        'episode_rewards': episode_rewards,
        'episode_infos': episode_infos,
    }

print("Evaluating trained model...")
print("-" * 40)

# Create fresh eval environment
eval_test_env = DummyVecEnv([make_env(0, CONFIG['seed'] + 2000)])

results = evaluate_model(model, eval_test_env, n_episodes=10)

print("-" * 40)
print(f"\nEvaluation Results:")
print(f"  Mean Reward: {results['mean_reward']:.2f} ± {results['std_reward']:.2f}")
print(f"  Mean Length: {results['mean_length']:.1f}")
print(f"  Best Episode: {max(results['episode_rewards']):.2f}")
print(f"  Worst Episode: {min(results['episode_rewards']):.2f}")

## 11. Visualize Episode

In [ ]:
def visualize_episode(model, env_config, max_steps=100):
    """
    Visualize a single episode with the trained model.
    """
    # Create environment with rendering support
    vis_env = FireControlEnv(
        swarm_size=env_config['swarm_size'],
        grid_size=env_config['grid_size'],
        max_episode_steps=max_steps,
        render_mode='rgb_array'
    )
    
    obs, info = vis_env.reset(seed=42)
    frames = []
    rewards = []
    
    # Wrap observation for model
    obs_wrapped = obs.reshape(1, -1)
    
    for step in range(max_steps):
        # Get action from model
        action, _ = model.predict(obs_wrapped, deterministic=True)
        action = action[0] if hasattr(action, '__len__') else action
        
        # Step environment
        obs, reward, terminated, truncated, info = vis_env.step(action)
        obs_wrapped = obs.reshape(1, -1)
        rewards.append(reward)
        
        # Render frame
        frame = vis_env.render()
        if frame is not None:
            frames.append(frame)
        
        if terminated or truncated:
            break
    
    vis_env.close()
    return frames, rewards, info

# Visualize an episode
print("Generating episode visualization...")
frames, rewards, final_info = visualize_episode(model, CONFIG, max_steps=50)

print(f"Episode completed in {len(rewards)} steps")
print(f"Total reward: {sum(rewards):.2f}")
print(f"Final state: {final_info}")

# Display frames
if len(frames) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    # Select frames to display
    indices = np.linspace(0, len(frames)-1, 8, dtype=int)
    
    for ax, idx in zip(axes, indices):
        ax.imshow(frames[idx])
        ax.set_title(f'Step {idx}')
        ax.axis('off')
    
    plt.suptitle('Episode Visualization', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No frames captured (rendering may not be supported in this environment)")

## 12. Compare with Baseline

In [ ]:
def evaluate_random_policy(env, n_episodes=10):
    """
    Evaluate random policy baseline.
    """
    episode_rewards = []
    
    for ep in range(n_episodes):
        obs = env.reset()
        done = False
        episode_reward = 0
        
        while not done:
            action = [env.action_space.sample()]
            obs, reward, done, info = env.step(action)
            episode_reward += reward[0]
        
        episode_rewards.append(episode_reward)
    
    return np.mean(episode_rewards), np.std(episode_rewards)

# Compare trained model with random baseline
print("Comparing trained model with random baseline...")
print("-" * 50)

# Random baseline
random_mean, random_std = evaluate_random_policy(eval_test_env, n_episodes=10)
print(f"Random Policy: {random_mean:.2f} ± {random_std:.2f}")

# Trained model
trained_mean = results['mean_reward']
trained_std = results['std_reward']
print(f"Trained Model: {trained_mean:.2f} ± {trained_std:.2f}")

# Improvement
improvement = ((trained_mean - random_mean) / abs(random_mean)) * 100 if random_mean != 0 else 0
print(f"\nImprovement: {improvement:.1f}%")

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
x = ['Random', 'Trained']
means = [random_mean, trained_mean]
stds = [random_std, trained_std]
colors = ['gray', 'green']

bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_ylabel('Mean Episode Reward')
ax.set_title('Policy Comparison')

# Add value labels
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{mean:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 13. Download Model (Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    import shutil
    
    # Create zip of checkpoints
    shutil.make_archive('fire_control_model', 'zip', CONFIG['checkpoint_dir'])
    
    print("Downloading trained model...")
    files.download('fire_control_model.zip')
else:
    print(f"Model saved in: {CONFIG['checkpoint_dir']}")
    print("Files:")
    for f in os.listdir(CONFIG['checkpoint_dir']):
        print(f"  - {f}")

## 14. TensorBoard (Optional)

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Launch TensorBoard
%tensorboard --logdir {CONFIG['log_dir']}

## 15. Cleanup

In [ ]:
# Close environments
train_env.close()
eval_env.close()
eval_test_env.close()

print("Environments closed.")
print("\n" + "="*60)
print("Training Complete!")
print("="*60)
print(f"\nSummary:")
print(f"  Algorithm: {CONFIG['algorithm']}")
print(f"  Total Timesteps: {CONFIG['total_timesteps']:,}")
print(f"  Episodes: {len(viz_callback.episode_rewards)}")
print(f"  Final Mean Reward: {results['mean_reward']:.2f}")
print(f"  Improvement over Random: {improvement:.1f}%")
print(f"\nModel saved to: {CONFIG['checkpoint_dir']}")

---

## Next Steps

1. **Increase training time**: Try `total_timesteps = 2_000_000` for better results
2. **Tune hyperparameters**: Adjust learning rate, batch size, etc.
3. **Try different algorithms**: Compare PPO, DQN, A2C performance
4. **Scale up**: Increase `swarm_size` and `grid_size` for more realistic scenarios
5. **Curriculum learning**: Start with easier scenarios and gradually increase difficulty

## Tips for Better Results

- Use **Colab Pro/Pro+** for longer training sessions and better GPUs
- **Save checkpoints frequently** to avoid losing progress
- **Monitor TensorBoard** to track training progress
- **Experiment with reward shaping** for faster convergence